# Process Mining Labor 1: Datenanalyse und Prozesserkennung

Dieses Notebook führt durch verschiedene Aufgaben der Datenanalyse mit Pandas und der Prozesserkennung mit PM4Py.

In [ ]:
# Installation der benötigten Bibliotheken aus der requirements.txt Datei
!pip install -r requirements.txt

# Aufgabe 1: Datenanalyse mit Pandas

In dieser Aufgabe lernen wir die Grundlagen der Datenanalyse mit Pandas kennen. Wir werden:
1. Einen DataFrame erstellen
2. Daten explorieren
3. Die Datenqualität prüfen
4. Initiale Analysen durchführen
5. Daten manipulieren
6. Statistische Analysen durchführen
7. Daten visualisieren

In [ ]:
# Import der benötigten Bibliotheken für die Datenanalyse
import pandas as pd  # Für Datenmanipulation und -analyse
import matplotlib.pyplot as plt  # Für Datenvisualisierung

## 1a) Pandas DataFrame

Ein Pandas DataFrame ist eine zweidimensionale, größenveränderbare und potentiell heterogene tabellarische Datenstruktur mit beschrifteten Achsen (Zeilen und Spalten). Es ist vergleichbar mit einer Tabelle in einer relationalen Datenbank oder einer Excel-Tabelle.

In diesem Beispiel erstellen wir einen DataFrame mit Fahrzeugdaten.

In [ ]:
# Erstellung eines Dictionary mit Fahrzeugdaten
vehicle_data = {
    'Fahrzeugtyp': ['Limousine', 'SUV', 'Kombi', 'Kleinwagen', 'Cabriolet'],
    'Kilometerstand': [54000, 23000, 120000, 89000, 45000],
    'Verbrauch_l/100km': [7.5, 9.8, 6.2, 4.9, 8.3],
    'Baujahr': [2018, 2020, 2015, 2017, 2019]
}

# Konvertierung des Dictionary in einen Pandas DataFrame
df_vehicle = pd.DataFrame(vehicle_data)

In [ ]:
# Ausgabe des erstellten DataFrames
print(df_vehicle)

## 1b) Datenexploration

Die Datenexploration ist ein wichtiger erster Schritt in der Datenanalyse. Hierbei geht es darum, einen Überblick über die Daten zu bekommen:
- Struktur der Daten verstehen
- Anzahl und Art der Variablen identifizieren
- Deskriptive Statistiken berechnen
- Datentypen prüfen

In diesem Beispiel untersuchen wir einen Datensatz mit Fahrzeugüberwachungsdaten.

In [ ]:
# Pfad zur CSV-Datei
file_path = './Daten/vehicle_monitor.csv'
# Einlesen der CSV-Datei in einen Pandas DataFrame mit Semikolon als Trennzeichen
vehicle_monitor = pd.read_csv(file_path, sep=';')

In [ ]:
print("Datenexploration:\n")
# Anzeige der ersten 5 Zeilen des DataFrames
print(vehicle_monitor.head(5))

# Ermittlung der Anzahl der Spalten
num_columns = vehicle_monitor.shape[1]
print(f'\nAnzahl der Spalten: {num_columns}')

# Namen der Spalten ausgeben
column_names = vehicle_monitor.columns.tolist()
print(f'\nNamen der Spalten: {column_names}')

In [ ]:
# Anzeige einer zusammenfassenden Information über den DataFrame
# Zeigt Spaltentypen, Nicht-Null-Werte und Speichernutzung
print(vehicle_monitor.info())

In [ ]:
# Berechnung deskriptiver Statistiken für numerische Spalten
# Zeigt min, max, Mittelwert, Standardabweichung und Perzentile
print(vehicle_monitor.describe())

In [ ]:
# Konvertierung der 'timestamp'-Spalte in das Datetime-Format für bessere Zeitanalysen
vehicle_monitor['timestamp'] = pd.to_datetime(vehicle_monitor['timestamp'])
# Überprüfung der Konvertierung durch erneutes Aufrufen von .info()
print(vehicle_monitor.info())

## 1c) Datenqualität prüfen

Eine wichtige Phase in der Datenanalyse ist die Überprüfung der Datenqualität. Hierbei werden:
- Fehlende Werte identifiziert
- Inkonsistenzen/Fehler in den Daten aufgedeckt
- Ausreißer erkannt
- Daten für die weitere Analyse vorbereitet

Wichtig: Schlechte Datenqualität kann zu fehlerhaften Analysen und falschen Schlussfolgerungen führen.

In [ ]:
# Zählen der fehlenden Werte pro Spalte
missing_values = vehicle_monitor.isnull().sum()
print("Fehlende Werte pro Spalte:\n", missing_values)

In [ ]:
'''
Erklärung des folgenden Codes:
    vehicle_monitor: Der Datensatz (Pandas DataFrame), der analysiert wird. In diesem Beispiel handelt es sich um einen DataFrame, der Fahrzeugdaten enthält.
    vehicle_monitor.isnull(): Diese Methode liefert einen DataFrame zurück, der die gleiche Struktur wie vehicle_monitor hat, aber für jede Zelle einen booleschen Wert (True oder False). True bedeutet, dass dieser Wert NaN oder None ist, also fehlt.
    vehicle_monitor.isnull().any(axis=1):
    any() ist eine Pandas-Methode, die überprüft, ob in einer bestimmten Dimension (axis=1 für Zeilen) mindestens ein True vorhanden ist. Das Ergebnis ist eine Pandas Series mit booleschen Werten: True bedeutet, dass mindestens ein Wert in der entsprechenden Zeile fehlt.
    axis=1 bedeutet, dass die Überprüfung zeilenweise erfolgt, d.h., es wird in jeder Zeile überprüft, ob eine der Spalten einen fehlenden Wert enthält.
    vehicle_monitor[vehicle_monitor.isnull().any(axis=1)]:
    Hier wird ein boolescher Index verwendet, um nur die Zeilen aus vehicle_monitor zurückzugeben, bei denen vehicle_monitor.isnull().any(axis=1) den Wert True hat. Das bedeutet, dass diese Zeilen mindestens einen fehlenden Wert enthalten.
    ['scene']:
    Nachdem die Zeilen mit fehlenden Werten gefiltert wurden, wird die Spalte scene extrahiert, um die Szenen zu erhalten, in denen diese fehlenden Werte auftreten.
    .unique():
    Die Methode unique() liefert eine Liste von einzigartigen Werten aus der Spalte scene. Damit werden alle eindeutigen Szenen ausgegeben, in denen mindestens ein Wert fehlt.
'''

# Identifizierung der eindeutigen Szenen, die fehlende Werte enthalten
missing_values_scenes = vehicle_monitor[vehicle_monitor.isnull().any(axis=1)]['scene'].unique()
print("\nSzenen mit fehlenden Werten:\n", missing_values_scenes.tolist())

In [ ]:
# Entfernen der Szenen mit fehlenden Werten aus dem DataFrame
scenes_to_remove = missing_values_scenes
vehicle_monitor = vehicle_monitor[~vehicle_monitor['scene'].isin(scenes_to_remove)]

In [ ]:
# Identifizierung ungültiger Daten: negative Geschwindigkeiten und Batteriestände
invalid_speed = vehicle_monitor[vehicle_monitor['vehicle_speed'] < 0]
invalid_battery = vehicle_monitor[vehicle_monitor['battery_level'] < 0]

# Ausgabe der Anzahl der ungültigen Datenpunkte
print(f'Anzahl negativer Geschwindigkeiten: {len(invalid_speed)}')
print(f'Anzahl negativer Batteriestände: {len(invalid_battery)}')

In [ ]:
# Identifizierung der Szenen mit ungültigen Daten
invalid_speed_scenes = vehicle_monitor[vehicle_monitor['vehicle_speed'] < 0]['scene'].unique()
invalid_battery_scenes = vehicle_monitor[vehicle_monitor['battery_level'] < 0]['scene'].unique()

# Ausgabe der Szenen-IDs mit ungültigen Daten
print(f'Szenen mit negativer Geschwindigkeit: {invalid_speed_scenes}')
print(f'Szenen mit negativem Batteriestand: {invalid_battery_scenes}')

In [ ]:
# Entfernen der Szenen mit negativen Batterieständen aus dem DataFrame
scenes_to_remove = invalid_battery_scenes
vehicle_monitor = vehicle_monitor[~vehicle_monitor['scene'].isin(scenes_to_remove)]

## 1d) Initiale Datenanalysen

Nach der Datenbereinigung können wir mit den ersten Analysen beginnen. Zunächst berechnen wir grundlegende statistische Kennzahlen, um einen ersten Eindruck von den Daten zu bekommen.

Diese Analysen helfen uns, Muster und potentielle Erkenntnisse in den Daten zu identifizieren.

In [ ]:
# Berechnung der durchschnittlichen Fahrzeuggeschwindigkeit
average_speed = vehicle_monitor['vehicle_speed'].mean()
print(f'Durchschnittliche Geschwindigkeit: {average_speed:.2f} km/h')

# Ermittlung der maximalen Fahrzeuggeschwindigkeit
max_speed = vehicle_monitor['vehicle_speed'].max()
print(f'Maximale Geschwindigkeit: {max_speed} km/h')

In [ ]:
# Berechnung der Kennzahlen für die Lenkgeschwindigkeit
average_steering_speed = vehicle_monitor['steering_speed'].mean()
max_steering_speed = vehicle_monitor['steering_speed'].max()
std_steering_speed = vehicle_monitor['steering_speed'].std()

# Ausgabe der berechneten Kennzahlen
print(f'Durchschnittliche Lenkgeschwindigkeit: {average_steering_speed:.2f} °/s')
print(f'Maximale Lenkgeschwindigkeit: {max_steering_speed} °/s')
print(f'Standardabweichung der Lenkgeschwindigkeit: {std_steering_speed:.2f} °/s')

## 1e) Datenmanipulation

Datenmanipulation umfasst die Transformation und Anreicherung der Daten. Dies kann die Erstellung neuer Variablen, die Umformung bestehender Daten oder die Kategorisierung kontinuierlicher Variablen beinhalten.

In diesem Beispiel kategorisieren wir die Fahrzeuggeschwindigkeit in drei Zustände:
- "stopped": Fahrzeug steht (Geschwindigkeit = 0)
- "slow": Fahrzeug fährt langsam (0 < Geschwindigkeit ≤ 20)
- "fast": Fahrzeug fährt schnell (Geschwindigkeit > 20)

In [ ]:
# Definition einer Funktion zur Klassifizierung des Fahrzeugzustands basierend auf der Geschwindigkeit
def classify_vehicle_state(speed):
    if speed == 0:
        return "stopped"
    elif 0 < speed <= 20:
        return "slow"
    else:
        return "fast"

In [ ]:
# Anwendung der Klassifizierungsfunktion auf die 'vehicle_speed'-Spalte
# Erstellung einer neuen Spalte 'vehicle_state' mit den kategorisierten Werten
vehicle_monitor['vehicle_state'] = vehicle_monitor['vehicle_speed'].apply(classify_vehicle_state)
# Anzeige der ersten 5 Zeilen mit der neuen Spalte
print(vehicle_monitor.head(5))

In [ ]:
# Filtern der Daten für Fahrzeugzustände "fast"
fast_vehicle_data = vehicle_monitor[vehicle_monitor['vehicle_state'] == "fast"]
# Ausgabe der Anzahl der Datenpunkte mit "fast" Geschwindigkeit
print(f'Anzahl der Datenpunkte mit "fast" Geschwindigkeit: {len(fast_vehicle_data)}')

## 1f) Statistische Analyse

Mit statistischen Analysen können wir komplexere Zusammenhänge in den Daten untersuchen. Dabei können wir:
- Gruppierte Statistiken berechnen
- Korrelationen identifizieren
- Muster und Trends erkennen

In diesem Beispiel untersuchen wir den Zusammenhang zwischen Fahrzeugzustand und Batteriestand sowie die Nutzung der Blinker bei hoher Geschwindigkeit.

In [ ]:
# Gruppierung der Daten nach 'vehicle_state' und Berechnung des durchschnittlichen Batteriestands
battery_means = vehicle_monitor.groupby('vehicle_state')['battery_level'].mean()
print('Durchschnittlicher Batteriestand pro Fahrzeuggeschwindigkeit:')
print(battery_means)

In [ ]:
# Untersuchung der Blinkernutzung bei "fast" Geschwindigkeit
# Summe der binären Werte (1/0) für linken und rechten Blinker
left_signal_count = fast_vehicle_data['left_signal'].sum()
right_signal_count = fast_vehicle_data['right_signal'].sum()
print(f'Häufigkeit der linken Blinker bei "fast": {left_signal_count}')
print(f'Häufigkeit der rechten Blinker bei "fast": {right_signal_count}')

## 1g) Visualisierung

Die Visualisierung ist ein mächtiges Werkzeug in der Datenanalyse, um Muster, Trends und Ausreißer schnell zu erkennen. Mit visuellen Darstellungen können komplexe Zusammenhänge oft besser verstanden werden als mit reinen Zahlen.

In diesem Beispiel visualisieren wir:
1. Die Verteilung des Batteriestands und der Reichweite
2. Den Geschwindigkeitsverlauf in zwei ausgewählten Szenen

In [ ]:
# Erstellung von Subplots mit zwei Histogrammen nebeneinander
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Erstes Histogramm für den Batteriestand
axes[0].hist(vehicle_monitor['battery_level'], bins=30, edgecolor='black', color='purple')
axes[0].set_xlabel('Batteriestand')
axes[0].set_ylabel('Anzahl der Datenpunkte')
axes[0].set_title('Verteilung des Batteriestands')

# Zweites Histogramm für die Reichweite
axes[1].hist(vehicle_monitor['available_distance'], bins=30, edgecolor='black', color='orange')
axes[1].set_xlabel('Reichweite')
axes[1].set_ylabel('Anzahl der Datenpunkte')
axes[1].set_title('Verteilung der Reichweite')

# Layout-Anpassungen
plt.tight_layout()
plt.show()

In [ ]:
# Auswahl der Daten für bestimmte Szenen
scene_id1 = 268
scene_data1 = vehicle_monitor[vehicle_monitor['scene'] == scene_id1]

# Die Szene 42 auswählen
scene_id2 = 42
scene_data2 = vehicle_monitor[vehicle_monitor['scene'] == scene_id2]

# Erstellung von Subplots mit zwei Diagrammen nebeneinander
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Erstes Diagramm (Szene 268): Geschwindigkeitsverlauf über die Zeit
axes[0].plot(scene_data1['timestamp'], scene_data1['vehicle_speed'], marker='o', color='blue')
axes[0].set_xlabel('Zeitpunkt')
axes[0].set_ylabel('Fahrzeuggeschwindigkeit (km/h)')
axes[0].set_title(f'Geschwindigkeitsverlauf der Szene {scene_id1}')
axes[0].tick_params(axis='x', rotation=45)  # Drehung der x-Achsen-Beschriftungen für bessere Lesbarkeit
axes[0].grid(True)  # Hinzufügen eines Gitters zur besseren Orientierung

# Zweites Diagramm (Szene 42): Geschwindigkeitsverlauf über die Zeit
axes[1].plot(scene_data2['timestamp'], scene_data2['vehicle_speed'], marker='o', color='green')
axes[1].set_xlabel('Zeitpunkt')
axes[1].set_ylabel('Fahrzeuggeschwindigkeit (km/h)')
axes[1].set_title(f'Geschwindigkeitsverlauf der Szene {scene_id2}')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True)

# Layout-Anpassungen
plt.tight_layout()
plt.show()

# Aufgabe 2: Prozess-Mining mit PM4Py

In dieser Aufgabe werden wir die PM4Py-Bibliothek nutzen, um Prozessdaten zu analysieren. Process Mining ist eine Technik, die Data Mining und Prozessanalyse kombiniert, um Einblicke in Geschäftsprozesse zu gewinnen.

Wir werden:
1. Prozessdaten einlesen
2. Einen Überblick über die Daten gewinnen
3. Den Prozess analysieren
4. Verschiedene Prozessmodelle vergleichen

In [ ]:
# Import der benötigten Bibliotheken für Process Mining
import pm4py  # Hauptbibliothek für Process Mining
import pandas as pd  # Für Datenmanipulation

## 2b) Prozessdaten einlesen

Process Mining verwendet Event-Logs als Grundlage für die Analyse. Ein Event-Log ist eine strukturierte Aufzeichnung von Ereignissen, die während der Ausführung eines Prozesses auftreten.

Das XES-Format (eXtensible Event Stream) ist ein standardisiertes XML-basiertes Format für Event-Logs und wird in PM4Py unterstützt.

In [ ]:
# Einlesen einer XES-Datei mit PM4Py
# XES ist ein standardisiertes Format für Event-Logs im Process Mining
process_data = pm4py.read_xes("./Daten/PrepaidTravelCost.xes")

In [ ]:
# Anzeige der ersten 5 Zeilen des eingelesenen Event-Logs
process_data.head(5)

## 2c) Überblick über die Daten

Bevor wir in die eigentliche Process-Mining-Analyse einsteigen, verschaffen wir uns einen Überblick über das Event-Log. Dies umfasst:
- Anzahl der Fälle (Cases)
- Anzahl der Ereignisse (Events)
- Anzahl der Attribute
- Datentypen
- Fehlende Werte

Diese grundlegende Datenexploration hilft uns, die Struktur und Qualität des Event-Logs zu verstehen.

In [ ]:
# Ermittlung der Anzahl eindeutiger Fälle (Cases) im Event-Log
# Ein Fall repräsentiert typischerweise eine Instanz des analysierten Prozesses
number_of_cases = len(process_data['case:concept:name'].unique())
print("Anzahl Fälle:", number_of_cases)

In [ ]:
# Ermittlung der Gesamtanzahl der Ereignisse (Events) im Event-Log
# Ein Ereignis repräsentiert eine Aktivität, die während eines Prozesses ausgeführt wurde
number_of_events = len(process_data)
print("Anzahl Ereignisse:", number_of_events)

In [ ]:
# Berechnung der durchschnittlichen Anzahl von Ereignissen pro Fall
# Dies gibt einen Hinweis auf die durchschnittliche Komplexität der Prozessinstanzen
percentage = round(number_of_events / number_of_cases, 1)
print("Durchschnittliche Anzahl Ereignisse pro Fall:", percentage)

In [ ]:
# Ermittlung der Anzahl der Spalten (Attribute) im Event-Log
number_of_attributes = len(process_data.columns)
print("Anzahl Spalten:", number_of_attributes)
# Ausgabe der Namen aller Spalten
print("\nSpaltennamen:", process_data.columns.tolist())

In [ ]:
# Anzeige einer zusammenfassenden Information über das Event-Log
# Zeigt Datentypen, Nicht-Null-Werte und Speichernutzung
process_data.info()

In [ ]:
# Zählen der fehlenden Werte pro Spalte im Event-Log
missing_values = process_data.isnull().sum()
missing_values  # Ausgabe der fehlenden Werte

## 2d) Überblick über den Prozess

Nach der allgemeinen Datenexploration analysieren wir nun den Prozess selbst. Dazu gehören:
- Identifizierung aller Aktivitäten
- Analyse der Start- und Endaktivitäten
- Identifizierung der beteiligten Rollen und Ressourcen
- Analyse der Prozessvarianten
- Berechnung von Prozessmetriken wie Falldauern

Diese Analysen geben uns erste Einblicke in den Ablauf und die Struktur des Prozesses.

In [ ]:
# Ermittlung aller eindeutigen Aktivitäten im Prozess
activities = process_data["concept:name"].unique()
print("Anzahl Aktivitäten:", len(activities), "\n")
print("Eindeutige Aktivitäten:\n")
for activity in activities:
    print(activity)

In [ ]:
# Ermittlung der Startaktivitäten und ihrer Häufigkeit
# Startaktivitäten sind die ersten Aktivitäten in jedem Fall
start_activities = pm4py.get_start_activities(process_data)
print("Aktivität : Anzahl\n")
for activity, number in start_activities.items():
    print(f"{activity}: {number}")

In [ ]:
# Ermittlung der Endaktivitäten und ihrer Häufigkeit
# Endaktivitäten sind die letzten Aktivitäten in jedem Fall
end_activities = pm4py.get_end_activities(process_data)
print("Aktivität : Anzahl\n")
for activity, number in end_activities.items():
    print(f"{activity}: {number}")

In [ ]:
# Ermittlung der eindeutigen involvierten Rollen im Prozess
roles = process_data['org:role'].unique()
print("Involvierte Rollen:", roles)
print("\nAnzahl unterschiedlicher Rollen:", len(roles))

In [ ]:
# Ermittlung der eindeutigen involvierten Ressourcen (z.B. Bearbeiter) im Prozess
resources = process_data['org:resource'].unique()
print("Involvierte Ressourcen:", resources)  # Korrektur der Ausgabebeschreibung
print("\nAnzahl unterschiedlicher Ressourcen:", len(resources))  # Korrektur der Ausgabebeschreibung

In [ ]:
# Ermittlung der Prozessvarianten und ihrer Häufigkeiten
# Eine Prozessvariante ist eine eindeutige Sequenz von Aktivitäten
variants = pm4py.stats.get_variants(process_data)
print("Anzahl unterschiedlicher Varianten:", len(variants))
print("\nAusführungen häufigster Variante:", max(variants.values()))

In [ ]:
# Ausgabe der ersten 5 häufigsten Prozessvarianten (Sequenzen von Aktivitäten)
for variant in list(variants.keys())[:5]:
    print(variant)

In [ ]:
# Berechnung der Dauer jedes Falls in Sekunden
case_durations = pm4py.stats.get_all_case_durations(process_data)
# Umrechnung der Falldauern von Sekunden in Stunden
case_durations = [duration/3600 for duration in case_durations]

# Berechnung der durchschnittlichen Falldauer in Stunden und Tagen
hours = round(sum(case_durations) / number_of_cases)
days = round(hours / 24)
print("Durchschnittliche Falldauer in Stunden:", hours)
print("Durchschnittliche Falldauer in Tagen:", days)

In [ ]:
# Identifizierung von Aktivitäten, die "REJECTED" im Namen enthalten
rejected_activities = [activity for activity in activities if "REJECTED" in activity]
print(rejected_activities)

In [ ]:
# Filtern des Event-Logs, um nur Events mit "REJECTED" Aktivitäten zu behalten
# Der Parameter retain=True gibt an, dass die gefilterten Aktivitäten beibehalten werden sollen
filtered_data = pm4py.filter_event_attribute_values(process_data,'concept:name', rejected_activities, retain=True)

In [ ]:
# Berechnung der Falldauern für das gefilterte Log mit REJECTED-Aktivitäten
case_durations_filtered_data = pm4py.stats.get_all_case_durations(filtered_data)
# Umrechnung der Falldauern von Sekunden in Stunden
case_durations_filtered_data = [duration/3600 for duration in case_durations_filtered_data]

# Anzahl der eindeutigen Fälle im gefilterten Log
number_of_cases_filtered = len(filtered_data['case:concept:name'].unique())

# Berechnung der durchschnittlichen Falldauer für Fälle mit REJECTED-Aktivitäten
hours = round(sum(case_durations_filtered_data) / number_of_cases_filtered)
days = round(hours / 24)
print("Durchschnittliche Falldauer mit 'REJECTED' in Stunden:", hours)
print("Durchschnittliche Falldauer mit 'REJECTED' in Tagen:", days)

## 2e) Prozessmodelle vergleichen

Process Discovery ist eine zentrale Technik im Process Mining, mit der automatisch Prozessmodelle aus Event-Logs erstellt werden können. In diesem Abschnitt:
- Erstellen wir ein Prozessmodell (Petri-Netz) für den gesamten Prozess
- Filtern wir das Event-Log, um ein vereinfachtes Modell zu erstellen
- Vergleichen wir die beiden Modelle

Dies hilft uns, verschiedene Perspektiven des Prozesses zu verstehen - von der umfassenden Gesamtansicht bis hin zu fokussierten Teilansichten.

In [ ]:
# Entdeckung eines Petri-Netzes aus dem gesamten Event-Log mit dem Alpha Miner
# Der Alpha Miner ist ein klassischer Algorithmus zur Process Discovery
petri_net, im, fm = pm4py.discover_petri_net_alpha(process_data)

In [ ]:
# Visualisierung des entdeckten Petri-Netzes
# im: initial marking (Anfangsmarkierung), fm: final marking (Endmarkierung)
pm4py.view_petri_net(petri_net,im,fm)

In [ ]:
# Filtern des Event-Logs, um nur die 2 häufigsten Prozessvarianten zu behalten
# Dies vereinfacht das resultierende Prozessmodell
filtered_data = pm4py.filter_variants_top_k(process_data, 2)

In [ ]:
# Entdeckung eines Petri-Netzes aus dem gefilterten Event-Log (Top 2 Varianten)
petri_net, im, fm = pm4py.discover_petri_net_alpha(filtered_data)

In [ ]:
# Visualisierung des vereinfachten Petri-Netzes (basierend auf den Top 2 Varianten)
pm4py.view_petri_net(petri_net,im,fm)

# Aufgabe 3: Erweiterte Prozessanalyse

In dieser Aufgabe vertiefen wir unsere Process-Mining-Kenntnisse durch die Analyse eines komplexeren Event-Logs. Wir werden:
1. Verschiedene Kennzahlen berechnen
2. Die Daten visualisieren
3. Fortgeschrittene Prozessmodelle erstellen und analysieren

Diese erweiterte Analyse gibt uns tiefere Einblicke in die Prozessleistung und -struktur.

## 3a) Kennzahlen berechnen

Für eine tiefere Analyse des Prozesses berechnen wir verschiedene Kennzahlen, die uns Einblicke in die Prozessleistung geben:
- Durchschnittliche Trace-Länge (Anzahl der Aktivitäten pro Fall)
- Falldauern
- Ressourcenauslastung

Diese Kennzahlen helfen uns, die Effizienz und Effektivität des Prozesses zu bewerten.

In [ ]:
# Import der benötigten Bibliotheken für die erweiterte Prozessanalyse
import pm4py
import pandas as pd
from sklearn.preprocessing import LabelEncoder  # Für die Kodierung von kategorialen Daten
import matplotlib.pyplot as plt

In [ ]:
# Einlesen eines weiteren, komplexeren XES-Event-Logs
log = pm4py.read_xes("./Daten/RequestforPayment.xes")
# Anzeige der ersten Zeilen des Logs
log.head()

In [ ]:
# Auswahl relevanter Spalten für die Analyse
# Wir fokussieren uns auf Case-ID, Zeitstempel, Aktivitätsname, Sublog, Konzeptname und Ressource
log_select = log[['case:concept:name', 'time:timestamp', 'concept:name', 'org:resource']]
log_select.head()

In [ ]:
# Generierung einer Übersicht über die Anzahl eindeutiger Werte pro Spalte
# Dies zeigt uns, wie viele verschiedene Fälle, Aktivitäten, Ressourcen usw. im Log enthalten sind
log_select.nunique()

In [ ]:
# Erstellung von Traces (Sequenzen von Aktivitäten pro Fall)

# Initialisierung des LabelEncoders zur Umwandlung von Aktivitätsnamen in numerische Werte
le = LabelEncoder()

# Kodierung der 'taskName'-Spalte und Speicherung in 'concept:name'
# Dies ermöglicht eine effizientere Verarbeitung der Aktivitätsnamen
log_select['task'] = le.fit_transform(log_select['concept:name'])
log_select.head()

In [ ]:
# Generierung eines Trace-Logs: Gruppierung nach Fall-ID und Auflistung der Aktivitäten pro Fall
# Ein Trace ist die Sequenz von Aktivitäten für einen bestimmten Fall
trace_log = log_select.groupby(['case:concept:name'])['task'].apply(list).reset_index()
trace_log.head()

In [ ]:
# Berechnung der Länge jedes Traces (Anzahl der Aktivitäten pro Fall)
# Dies gibt uns Einblick in die Komplexität der einzelnen Prozessinstanzen
trace_log["trace_length"] = trace_log["task"].apply(lambda x: len(x))
trace_log

In [ ]:
# Berechnung der durchschnittlichen Trace-Länge
# Dies zeigt die durchschnittliche Anzahl von Aktivitäten, die in einer Prozessinstanz ausgeführt werden
trace_log["trace_length"].mean()

In [ ]:
# Konvertierung der 'concept:name'-Spalte in Strings
# Dies ist wichtig für die PM4Py-Funktionen, die String-basierte Aktivitätsnamen erwarten
log_select["concept:name"] = log_select["concept:name"].apply(lambda x: str(x))
log_select.head()

In [ ]:
# Berechnung aller Falldauern in Sekunden
# Die Falldauer ist die Zeit von der ersten bis zur letzten Aktivität eines Falls
case_durations = pm4py.get_all_case_durations(log_select)
case_durations

In [ ]:
# Entfernen des ersten (möglicherweise fehlerhaften) Werts aus der Liste der Falldauern
case_durations.pop(0)

# Berechnung der durchschnittlichen Falldauer in Sekunden
average_duration_seconds = sum(case_durations)/len(case_durations)
print(f"Durchschnittliche Falldauer in Sekunden: {average_duration_seconds}")

In [ ]:
# Umrechnung der Falldauern von Sekunden in Tage für eine bessere Interpretierbarkeit
case_durations = [(duration / 3600)/24 for duration in case_durations]

## 3b) Visualisierung

Die Visualisierung hilft uns, Muster und Trends in den Prozessdaten besser zu erkennen. Wir erstellen zwei wichtige Visualisierungen:
1. Histogramm der Anzahl von Aktivitäten pro Fall - zeigt die Verteilung der Prozesskomplexität
2. Histogramm der Falldauern - zeigt die Verteilung der Prozessdurchlaufzeiten

Diese Visualisierungen ermöglichen es uns, Ausreißer und Anomalien im Prozess zu identifizieren.

In [ ]:
# Berechnung der Anzahl von Aktivitäten pro Fall
number_activities_per_case = log_select.groupby('case:concept:name').size()

# Erstellung eines Histogramms der Anzahl von Aktivitäten pro Fall
plt.figure(figsize=(8, 6))
number_activities_per_case.plot(kind='hist', bins=range(1, number_activities_per_case.max() + 2), edgecolor='black')
plt.xlabel('Anzahl Aktivitäten pro Case')
plt.ylabel('Anzahl der Cases')
plt.title('Verteilung der Anzahl Aktivitäten pro Case')
plt.grid(axis='y')
plt.show()

In [ ]:
# Erstellung eines Histogramms der Falldauern (in Tagen)
plt.figure(figsize=(8, 6))
plt.hist(case_durations, bins=100, edgecolor='black')
plt.xlabel('Dauer pro Case (in Tagen)')  # Korrigierte Beschriftung
plt.ylabel('Anzahl der Cases')
plt.title('Verteilung der Dauer pro Case')
plt.grid(axis='y')
plt.show()

## 3c) Prozessmodell

In diesem Abschnitt erstellen wir verschiedene Prozessmodelle:
1. Ein Modell für den gesamten Prozess
2. Ein Modell für einen Teilprozess (basierend auf einem Sublog)
3. Ein Modell für die häufigsten Varianten des Teilprozesses

Durch die Erstellung verschiedener Modelle können wir den Prozess aus unterschiedlichen Perspektiven betrachten und sowohl die Gesamtstruktur als auch spezifische Teilaspekte analysieren.

In [ ]:
# Entdeckung eines Petri-Netzes aus dem gesamten Event-Log mit dem Alpha Miner
petri_net, im, fm = pm4py.discover_petri_net_alpha(log_select)

# Visualisierung des entdeckten Petri-Netzes
pm4py.view_petri_net(petri_net, im, fm)

In [ ]:
# Anzeige von Informationen über den Event-Log
# Dies hilft uns, die Struktur und Eigenschaften des Logs besser zu verstehen
log_select.info()

In [ ]:
variants = pm4py.stats.get_variants(log_select)
print("Anzahl unterschiedlicher Varianten:", len(variants))
print("\nAusführungen häufigster Variante:", max(variants.values()))

In [ ]:
# Filtern des Event-Logs, um nur einen bestimmten Teilprozess (Sublog) zu betrachten
filtered_log = pm4py.filter_variants_by_coverage_percentage(
    log_select,
    0.2)
# Anzeige von Informationen über den gefilterten Log
filtered_log.info()

In [ ]:
# Entdeckung eines Petri-Netzes für die Top-5-Varianten des Teilprozesses
petri_net, im, fm = pm4py.discover_petri_net_alpha(filtered_log)

In [ ]:
# Visualisierung des Petri-Netzes für die Top-5-Varianten
# Korrektur: pm4py.vis.view_petri_net sollte pm4py.view_petri_net sein
pm4py.view_petri_net(petri_net, im, fm)

In [ ]:
# Überprüfung, ob das erstellte Petri-Netz ein Workflow-Netz (WF-Net) ist
# Ein Workflow-Netz ist ein spezielles Petri-Netz mit definierten Start- und Endpunkten
is_workflow_net = pm4py.check_is_workflow_net(petri_net)
print(f"Ist das Petri-Netz ein Workflow-Netz? {is_workflow_net}")

In [ ]:
# Ausgabe zusätzlicher Informationen zu den Top-5-Varianten
# Ermittlung der Start- und Endaktivitäten der häufigsten Prozessvarianten
start_activities_top_k = pm4py.get_start_activities(filtered_log)
print("Startaktivitäten:", start_activities_top_k)

end_activities_top_k = pm4py.get_end_activities(filtered_log)
print("Endaktivitäten:", end_activities_top_k)